# 🧠 Neural Networks Training Pipeline

## Welcome to the Trading ML Training Notebook!

This notebook provides a **complete workflow** for training neural network models for stock market prediction. Whether you're a beginner or experienced practitioner, this guided pipeline will help you:

- 📊 **Fetch & preprocess** market data from multiple sources
- 🤖 **Train** LSTM, DNN, and Transformer models
- 📈 **Evaluate** model performance with backtesting
- 🚀 **Export** models to ONNX for dashboard deployment
- 💾 **Auto-save** checkpoints to resume interrupted sessions

### Supported Models
| Model | Best For | Training Time |
|-------|----------|---------------|
| **LSTM** | Time series with long-term dependencies | Medium |
| **DNN** | Fast baseline predictions | Fast |
| **Transformer** | Complex pattern recognition | Slow |

### How to Use This Notebook
1. **Run cells in order** (top to bottom)
2. **Green checkmarks** ✅ indicate completed steps
3. **Interactive widgets** let you configure without coding
4. **Checkpoints** auto-save - you can close and resume anytime!

---
*💡 Tip: Hover over any parameter widget for helpful explanations!*

## 📦 Section 1: First-Time Setup

This section handles the initial environment setup. **Run this once** when you first open the notebook.

### What happens here:
1. ✅ Install required packages (PyTorch, pandas, etc.)
2. ✅ Mount Google Drive for checkpoints
3. ✅ Configure API keys for data sources
4. ✅ Verify everything is working

In [ ]:
#@title 🔧 Step 1.1: Install Dependencies { display-mode: "form" }
#@markdown Click the **Run** button (▶️) to install all required packages.
#@markdown This may take 2-3 minutes on first run.

import subprocess
import sys

def install_packages():
    """Install required packages with progress tracking."""
    packages = [
        ("torch", "PyTorch - Deep learning framework"),
        ("pandas", "Pandas - Data manipulation"),
        ("numpy", "NumPy - Numerical computing"),
        ("yfinance", "yFinance - Yahoo Finance data"),
        ("scikit-learn", "Scikit-learn - ML utilities"),
        ("matplotlib", "Matplotlib - Plotting"),
        ("seaborn", "Seaborn - Statistical visualization"),
        ("ipywidgets", "ipywidgets - Interactive widgets"),
        ("onnx", "ONNX - Model export format"),
        ("onnxruntime", "ONNX Runtime - Model validation"),
        ("optuna", "Optuna - Hyperparameter tuning"),
        ("wandb", "Weights & Biases - Experiment tracking"),
        ("h5py", "H5py - Checkpoint storage"),
        ("requests", "Requests - API calls"),
        ("tqdm", "tqdm - Progress bars"),
    ]
    
    print("📦 Installing packages...\n")
    print("=" * 60)
    
    for i, (package, description) in enumerate(packages, 1):
        print(f"[{i}/{len(packages)}] Installing {package}...")
        print(f"    📝 {description}")
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", package],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
            print(f"    ✅ Success!\n")
        except subprocess.CalledProcessError:
            print(f"    ⚠️ Warning: Could not install {package}")
            print(f"    💡 Try: !pip install {package}\n")
    
    print("=" * 60)
    print("\n✅ Package installation complete!")
    print("💡 If any package failed, you can install it manually above.")

# Run installation
install_packages()

In [ ]:
#@title 💾 Step 1.2: Mount Google Drive { display-mode: "form" }
#@markdown Your checkpoints, data, and exported models will be saved to Google Drive.
#@markdown This ensures you never lose progress!

import os

# Check if running in Colab
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IN_COLAB:
    from google.colab import drive
    
    print("🔗 Mounting Google Drive...")
    print("   You may need to authorize access in the popup window.\n")
    
    try:
        drive.mount('/content/drive')
        
        # Create project directory structure
        DRIVE_BASE = '/content/drive/MyDrive/trading_ml'
        DIRS = {
            'checkpoints': f'{DRIVE_BASE}/checkpoints',
            'data': f'{DRIVE_BASE}/data',
            'models': f'{DRIVE_BASE}/models',
            'exports': f'{DRIVE_BASE}/exports',
            'logs': f'{DRIVE_BASE}/logs'
        }
        
        for name, path in DIRS.items():
            os.makedirs(path, exist_ok=True)
            print(f"✅ Created {name}: {path}")
        
        print("\n" + "=" * 60)
        print("✅ Google Drive mounted successfully!")
        print(f"📁 Project folder: {DRIVE_BASE}")
        
    except Exception as e:
        print(f"❌ Error mounting Drive: {e}")
        print("💡 Try: Runtime → Restart runtime, then run this cell again")
else:
    # Local environment - use local folders
    DRIVE_BASE = './trading_ml_data'
    DIRS = {
        'checkpoints': f'{DRIVE_BASE}/checkpoints',
        'data': f'{DRIVE_BASE}/data',
        'models': f'{DRIVE_BASE}/models',
        'exports': f'{DRIVE_BASE}/exports',
        'logs': f'{DRIVE_BASE}/logs'
    }
    
    for name, path in DIRS.items():
        os.makedirs(path, exist_ok=True)
    
    print("📁 Running in local environment")
    print(f"📂 Data folder: {DRIVE_BASE}")
    print("✅ Directories created!")

In [ ]:
#@title 🔑 Step 1.3: Configure API Keys { display-mode: "form" }
#@markdown API keys allow access to premium data sources and experiment tracking.
#@markdown 
#@markdown **In Colab**: Add secrets using the 🔑 icon in the left sidebar
#@markdown **Locally**: Set environment variables or enter below

import os

# Initialize API keys dictionary
API_KEYS = {}

def get_api_key(name, env_var, required=False):
    """Safely get API key from multiple sources."""
    key = None
    
    # Try Colab secrets first
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get(env_var)
        except:
            pass
    
    # Try environment variable
    if not key:
        key = os.environ.get(env_var)
    
    # Status message
    if key:
        masked = key[:4] + '*' * (len(key) - 8) + key[-4:] if len(key) > 8 else '****'
        print(f"✅ {name}: {masked}")
        return key
    else:
        status = "❌ Missing (Required)" if required else "⚪ Not set (Optional)"
        print(f"{status}: {name}")
        if required:
            print(f"   💡 Add '{env_var}' to Colab Secrets or set environment variable")
        return None

print("🔑 Checking API Keys...\n")
print("=" * 60)

# Check each API key
API_KEYS['alpha_vantage'] = get_api_key(
    "Alpha Vantage", 
    "ALPHA_VANTAGE_KEY",
    required=False
)

API_KEYS['dashboard'] = get_api_key(
    "Dashboard API", 
    "DASHBOARD_API_KEY",
    required=False
)

API_KEYS['wandb'] = get_api_key(
    "Weights & Biases",
    "WANDB_API_KEY", 
    required=False
)

print("=" * 60)
print("\n📋 Summary:")
print("   • Yahoo Finance: ✅ No API key needed (free)")
print("   • Alpha Vantage: " + ("✅ Configured" if API_KEYS['alpha_vantage'] else "⚪ Using Yahoo Finance instead"))
print("   • W&B Tracking:  " + ("✅ Configured" if API_KEYS['wandb'] else "⚪ Using local logging"))
print("   • Dashboard API: " + ("✅ Configured" if API_KEYS['dashboard'] else "⚪ Manual upload required"))

print("\n💡 You can proceed without optional API keys - we'll use free alternatives!")

In [ ]:
#@title ✅ Step 1.4: Verify Environment { display-mode: "form" }
#@markdown This cell runs health checks to ensure everything is working.

import torch
import numpy as np

def run_health_checks():
    """Run comprehensive environment health checks."""
    results = {}
    
    print("🔍 Running Health Checks...\n")
    print("=" * 60)
    
    # Check 1: PyTorch
    try:
        print("1️⃣ PyTorch Installation")
        print(f"   Version: {torch.__version__}")
        results['pytorch'] = True
        print("   ✅ Working!\n")
    except Exception as e:
        results['pytorch'] = False
        print(f"   ❌ Error: {e}\n")
    
    # Check 2: GPU/CUDA
    print("2️⃣ GPU Availability")
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"   🎮 GPU: {gpu_name}")
        print(f"   💾 Memory: {gpu_memory:.1f} GB")
        results['gpu'] = True
        print("   ✅ GPU acceleration available!\n")
    else:
        print("   ⚪ No GPU detected - will use CPU")
        print("   💡 Training will be slower but still work")
        results['gpu'] = False
        print("   ⚠️ Consider enabling GPU: Runtime → Change runtime type → GPU\n")
    
    # Check 3: Data libraries
    print("3️⃣ Data Libraries")
    try:
        import pandas as pd
        import yfinance as yf
        print(f"   Pandas: {pd.__version__}")
        print(f"   yfinance: {yf.__version__}")
        results['data_libs'] = True
        print("   ✅ Ready!\n")
    except ImportError as e:
        results['data_libs'] = False
        print(f"   ❌ Missing: {e}\n")
    
    # Check 4: Storage access
    print("4️⃣ Storage Access")
    try:
        test_file = os.path.join(DIRS['checkpoints'], '.test_write')
        with open(test_file, 'w') as f:
            f.write('test')
        os.remove(test_file)
        results['storage'] = True
        print(f"   📁 Checkpoint folder: {DIRS['checkpoints']}")
        print("   ✅ Read/write access confirmed!\n")
    except Exception as e:
        results['storage'] = False
        print(f"   ❌ Storage error: {e}\n")
    
    # Check 5: Network (quick test)
    print("5️⃣ Network Connectivity")
    try:
        import urllib.request
        urllib.request.urlopen('https://finance.yahoo.com', timeout=5)
        results['network'] = True
        print("   ✅ Internet connection working!\n")
    except:
        results['network'] = False
        print("   ⚠️ Network issues detected\n")
    
    # Summary
    print("=" * 60)
    passed = sum(results.values())
    total = len(results)
    
    if passed == total:
        print(f"\n🎉 All {total} checks passed! You're ready to train.")
    elif passed >= total - 1:
        print(f"\n✅ {passed}/{total} checks passed. Ready to proceed!")
    else:
        print(f"\n⚠️ {passed}/{total} checks passed. Review warnings above.")
    
    return results

# Run the checks
health_results = run_health_checks()

# Set device for training
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️ Training device: {DEVICE}")

## 🔄 Section 2: Session Resume

This section checks for previous work and lets you continue where you left off.

**How it works:**
- Checkpoints are automatically saved after each major step
- If your session disconnects, your progress is preserved
- You can choose to resume or start fresh

In [ ]:
#@title 🔍 Check for Previous Session { display-mode: "form" }
#@markdown Automatically detects and offers to resume from checkpoints.

import glob
import json
from datetime import datetime
import h5py

class SessionManager:
    """Manages session state and checkpoints."""
    
    def __init__(self, checkpoint_dir):
        self.checkpoint_dir = checkpoint_dir
        self.current_session = None
        self.state = {
            'step': 'setup',
            'config': {},
            'data_loaded': False,
            'preprocessed': False,
            'models_trained': [],
            'timestamp': None
        }
    
    def find_checkpoints(self):
        """Find all available checkpoints."""
        pattern = os.path.join(self.checkpoint_dir, 'session_*.h5')
        checkpoints = glob.glob(pattern)
        
        if not checkpoints:
            return []
        
        # Sort by modification time (newest first)
        checkpoints.sort(key=os.path.getmtime, reverse=True)
        
        # Get metadata for each
        checkpoint_info = []
        for cp in checkpoints:
            try:
                with h5py.File(cp, 'r') as f:
                    info = {
                        'path': cp,
                        'timestamp': f.attrs.get('timestamp', 'Unknown'),
                        'step': f.attrs.get('step', 'Unknown'),
                        'models': list(f.attrs.get('models_trained', [])),
                    }
                    checkpoint_info.append(info)
            except:
                continue
        
        return checkpoint_info
    
    def load_checkpoint(self, path):
        """Load state from checkpoint file."""
        with h5py.File(path, 'r') as f:
            self.state = {
                'step': f.attrs.get('step', 'setup'),
                'config': json.loads(f.attrs.get('config', '{}')),
                'data_loaded': f.attrs.get('data_loaded', False),
                'preprocessed': f.attrs.get('preprocessed', False),
                'models_trained': list(f.attrs.get('models_trained', [])),
                'timestamp': f.attrs.get('timestamp', None)
            }
            
            # Load data arrays if present
            self.data = {}
            for key in ['X_train', 'X_test', 'y_train', 'y_test']:
                if key in f:
                    self.data[key] = f[key][:]
        
        print(f"✅ Loaded checkpoint from {self.state['timestamp']}")
        return self.state
    
    def save_checkpoint(self, step, **kwargs):
        """Save current state to checkpoint."""
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'session_{timestamp}.h5'
        path = os.path.join(self.checkpoint_dir, filename)
        
        self.state['step'] = step
        self.state['timestamp'] = timestamp
        self.state.update(kwargs)
        
        with h5py.File(path, 'w') as f:
            # Save metadata
            f.attrs['step'] = step
            f.attrs['timestamp'] = timestamp
            f.attrs['config'] = json.dumps(self.state.get('config', {}))
            f.attrs['data_loaded'] = self.state.get('data_loaded', False)
            f.attrs['preprocessed'] = self.state.get('preprocessed', False)
            f.attrs['models_trained'] = self.state.get('models_trained', [])
            
            # Save data arrays if available
            if hasattr(self, 'data'):
                for key, arr in self.data.items():
                    f.create_dataset(key, data=arr, compression='gzip')
        
        print(f"💾 Checkpoint saved: {filename}")
        return path

# Initialize session manager
session = SessionManager(DIRS['checkpoints'])

# Check for existing checkpoints
print("🔍 Checking for previous sessions...\n")
checkpoints = session.find_checkpoints()

if checkpoints:
    print("=" * 60)
    print("📋 Found previous sessions:\n")
    
    for i, cp in enumerate(checkpoints[:5], 1):  # Show top 5
        print(f"  {i}. {cp['timestamp']}")
        print(f"     Step: {cp['step']}")
        print(f"     Models trained: {len(cp['models'])}")
        print()
    
    print("=" * 60)
    print("\n🤔 Would you like to resume?")
    print("   • Run the next cell to RESUME from the latest checkpoint")
    print("   • Or skip to Section 3 to START FRESH")
    
    LATEST_CHECKPOINT = checkpoints[0]['path']
else:
    print("✨ No previous sessions found - starting fresh!")
    print("   Continue to Section 3 to configure your experiment.")
    LATEST_CHECKPOINT = None

In [ ]:
#@title 🔄 Resume from Checkpoint (Optional) { display-mode: "form" }
#@markdown **Only run this cell if you want to resume a previous session.**
#@markdown Skip this cell to start a fresh experiment.

if LATEST_CHECKPOINT:
    print("🔄 Resuming from checkpoint...\n")
    
    state = session.load_checkpoint(LATEST_CHECKPOINT)
    
    print("\n📊 Restored Session State:")
    print(f"   • Last step: {state['step']}")
    print(f"   • Data loaded: {'✅' if state['data_loaded'] else '❌'}")
    print(f"   • Preprocessed: {'✅' if state['preprocessed'] else '❌'}")
    print(f"   • Models trained: {len(state['models_trained'])}")
    
    if state['config']:
        print(f"\n⚙️ Configuration:")
        for key, value in state['config'].items():
            print(f"   • {key}: {value}")
    
    print("\n✅ Session restored! Continue from the appropriate section.")
else:
    print("⚠️ No checkpoint to resume from.")
    print("   Continue to Section 3 to configure a new experiment.")

## ⚙️ Section 3: Experiment Configuration

Use the interactive widgets below to configure your experiment **without writing any code!**

### Configuration Options:
- **Model Selection**: Choose which neural network(s) to train
- **Data Settings**: Select data source, assets, and date range
- **Hyperparameters**: Customize learning rate, epochs, batch size, etc.
- **Display Mode**: Verbose (detailed) or Quiet (minimal) output

In [ ]:
#@title 🎛️ Interactive Configuration Panel { display-mode: "form" }
#@markdown Adjust the settings below, then run this cell to save your configuration.

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ============================================================
# CONFIGURATION WIDGETS
# ============================================================

# Store configuration
CONFIG = {}

# --- Basic Settings ---
experiment_name = widgets.Text(
    value=f'exp_{datetime.now().strftime("%Y%m%d_%H%M")}',
    description='Experiment:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

model_select = widgets.SelectMultiple(
    options=['LSTM', 'DNN', 'Transformer'],
    value=['LSTM'],
    description='Models:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px', height='80px')
)

verbose_mode = widgets.ToggleButtons(
    options=['Verbose', 'Quiet'],
    value='Verbose',
    description='Output:',
    style={'description_width': '100px'}
)

# --- Data Settings ---
data_source = widgets.Dropdown(
    options=[
        ('Yahoo Finance (Free)', 'yahoo'),
        ('Alpha Vantage (API Key)', 'alpha_vantage'),
        ('NSE/BSE India', 'nse_bse')
    ],
    value='yahoo',
    description='Data Source:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

ticker_input = widgets.Text(
    value='AAPL',
    description='Ticker(s):',
    placeholder='AAPL, MSFT, GOOGL',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

start_date = widgets.DatePicker(
    description='Start Date:',
    value=datetime(2020, 1, 1).date(),
    style={'description_width': '100px'}
)

end_date = widgets.DatePicker(
    description='End Date:',
    value=datetime(2024, 12, 31).date(),
    style={'description_width': '100px'}
)

train_split = widgets.FloatSlider(
    value=0.8,
    min=0.5,
    max=0.95,
    step=0.05,
    description='Train Split:',
    style={'description_width': '100px'},
    readout_format='.0%'
)

# --- Model Hyperparameters ---
lookback = widgets.IntSlider(
    value=60,
    min=10,
    max=200,
    step=10,
    description='Lookback:',
    style={'description_width': '100px'}
)

epochs = widgets.IntSlider(
    value=100,
    min=10,
    max=500,
    step=10,
    description='Epochs:',
    style={'description_width': '100px'}
)

batch_size = widgets.Dropdown(
    options=[16, 32, 64, 128, 256],
    value=32,
    description='Batch Size:',
    style={'description_width': '100px'}
)

learning_rate = widgets.SelectionSlider(
    options=[0.0001, 0.0005, 0.001, 0.005, 0.01],
    value=0.001,
    description='Learn Rate:',
    style={'description_width': '100px'}
)

hidden_dim = widgets.Dropdown(
    options=[32, 64, 128, 256],
    value=64,
    description='Hidden Dim:',
    style={'description_width': '100px'}
)

num_layers = widgets.IntSlider(
    value=2,
    min=1,
    max=4,
    description='Layers:',
    style={'description_width': '100px'}
)

dropout = widgets.FloatSlider(
    value=0.2,
    min=0,
    max=0.5,
    step=0.1,
    description='Dropout:',
    style={'description_width': '100px'}
)

# --- Output Area ---
output = widgets.Output()

# ============================================================
# LAYOUT
# ============================================================

# Section headers
def header(text, emoji='📌'):
    return widgets.HTML(f'<h4 style="margin-top:15px;">{emoji} {text}</h4>')

# Tooltips
tooltips = {
    'lookback': '📝 Number of past days to use for prediction. Higher = more context but slower.',
    'epochs': '📝 Training iterations. More epochs = better learning but risk of overfitting.',
    'batch_size': '📝 Samples per training step. Larger = faster but uses more memory.',
    'learning_rate': '📝 How fast the model learns. Too high = unstable, too low = slow.',
    'hidden_dim': '📝 Model capacity. Larger = can learn more complex patterns.',
    'dropout': '📝 Regularization to prevent overfitting. Higher = more regularization.',
}

# Build the configuration panel
config_panel = widgets.VBox([
    widgets.HTML('<h3>🎛️ Experiment Configuration</h3>'),
    widgets.HTML('<hr>'),
    
    header('Basic Settings', '📋'),
    experiment_name,
    model_select,
    verbose_mode,
    
    header('Data Settings', '📊'),
    data_source,
    ticker_input,
    widgets.HBox([start_date, end_date]),
    train_split,
    
    header('Model Hyperparameters', '🔧'),
    widgets.HTML(f'<small style="color:gray;">{tooltips["lookback"]}</small>'),
    lookback,
    widgets.HTML(f'<small style="color:gray;">{tooltips["epochs"]}</small>'),
    epochs,
    widgets.HTML(f'<small style="color:gray;">{tooltips["batch_size"]}</small>'),
    batch_size,
    widgets.HTML(f'<small style="color:gray;">{tooltips["learning_rate"]}</small>'),
    learning_rate,
    widgets.HTML(f'<small style="color:gray;">{tooltips["hidden_dim"]}</small>'),
    hidden_dim,
    num_layers,
    widgets.HTML(f'<small style="color:gray;">{tooltips["dropout"]}</small>'),
    dropout,
    
    widgets.HTML('<hr>'),
    output
])

# Display the panel
display(config_panel)

# ============================================================
# SAVE CONFIGURATION
# ============================================================

def save_config():
    """Save current widget values to CONFIG."""
    global CONFIG
    CONFIG = {
        'experiment_name': experiment_name.value,
        'models': list(model_select.value),
        'verbose': verbose_mode.value == 'Verbose',
        'data_source': data_source.value,
        'tickers': [t.strip() for t in ticker_input.value.split(',')],
        'start_date': start_date.value.strftime('%Y-%m-%d'),
        'end_date': end_date.value.strftime('%Y-%m-%d'),
        'train_split': train_split.value,
        'lookback': lookback.value,
        'epochs': epochs.value,
        'batch_size': batch_size.value,
        'learning_rate': learning_rate.value,
        'hidden_dim': hidden_dim.value,
        'num_layers': num_layers.value,
        'dropout': dropout.value
    }
    return CONFIG

# Auto-save on widget change
for w in [experiment_name, model_select, verbose_mode, data_source, ticker_input,
          start_date, end_date, train_split, lookback, epochs, batch_size,
          learning_rate, hidden_dim, num_layers, dropout]:
    w.observe(lambda _: save_config(), names='value')

# Initial save
CONFIG = save_config()

with output:
    print("✅ Configuration panel loaded!")
    print("   Adjust settings above, then run the next cell to confirm.")

In [ ]:
#@title ✅ Confirm Configuration { display-mode: "form" }
#@markdown Review and confirm your experiment settings.

# Update CONFIG with latest values
CONFIG = save_config()

print("=" * 60)
print("📋 EXPERIMENT CONFIGURATION SUMMARY")
print("=" * 60)

print(f"\n🏷️  Experiment: {CONFIG['experiment_name']}")
print(f"🤖 Models: {', '.join(CONFIG['models'])}")
print(f"📢 Mode: {'Verbose (detailed output)' if CONFIG['verbose'] else 'Quiet (minimal output)'}")

print(f"\n📊 DATA SETTINGS:")
print(f"   Source: {CONFIG['data_source']}")
print(f"   Tickers: {', '.join(CONFIG['tickers'])}")
print(f"   Date Range: {CONFIG['start_date']} to {CONFIG['end_date']}")
print(f"   Train/Test Split: {CONFIG['train_split']:.0%} / {1-CONFIG['train_split']:.0%}")

print(f"\n🔧 MODEL HYPERPARAMETERS:")
print(f"   Lookback Period: {CONFIG['lookback']} days")
print(f"   Epochs: {CONFIG['epochs']}")
print(f"   Batch Size: {CONFIG['batch_size']}")
print(f"   Learning Rate: {CONFIG['learning_rate']}")
print(f"   Hidden Dimension: {CONFIG['hidden_dim']}")
print(f"   Number of Layers: {CONFIG['num_layers']}")
print(f"   Dropout: {CONFIG['dropout']}")

print("\n" + "=" * 60)

# Save to session
session.state['config'] = CONFIG
session.save_checkpoint('configuration', config=CONFIG)

print("\n✅ Configuration saved!")
print("   Proceed to Section 4 to fetch data.")

## 📊 Section 4: Data Acquisition

This section fetches historical market data from your configured source.

### What happens:
1. 📡 Connect to data API (Yahoo Finance, Alpha Vantage, etc.)
2. ⏳ Download data with rate limiting (automatic retry on limits)
3. ✅ Validate data quality (check for gaps, missing values)
4. 💾 Save checkpoint with raw data

**Rate Limiting**: If we hit API limits, the notebook will automatically wait and retry. You'll see a countdown timer.

In [ ]:
#@title 📡 Fetch Market Data { display-mode: "form" }
#@markdown Downloads historical data with automatic rate limiting and retry.

import pandas as pd
import yfinance as yf
import time
from tqdm.notebook import tqdm

class RateLimiter:
    """Simple rate limiter with exponential backoff."""
    
    def __init__(self, calls_per_minute=5):
        self.calls_per_minute = calls_per_minute
        self.min_interval = 60.0 / calls_per_minute
        self.last_call = 0
        self.backoff = 1
        
    def wait(self):
        """Wait if needed to respect rate limits."""
        elapsed = time.time() - self.last_call
        if elapsed < self.min_interval * self.backoff:
            wait_time = self.min_interval * self.backoff - elapsed
            if CONFIG['verbose']:
                print(f"   ⏳ Rate limit: waiting {wait_time:.1f}s...")
            time.sleep(wait_time)
        self.last_call = time.time()
        
    def success(self):
        """Reset backoff on success."""
        self.backoff = 1
        
    def error(self):
        """Increase backoff on error."""
        self.backoff = min(self.backoff * 2, 60)


def fetch_yahoo_data(ticker, start, end, rate_limiter):
    """Fetch data from Yahoo Finance with rate limiting."""
    rate_limiter.wait()
    
    try:
        stock = yf.Ticker(ticker)
        df = stock.history(start=start, end=end)
        
        if df.empty:
            raise ValueError(f"No data returned for {ticker}")
        
        # Standardize columns
        df.columns = [c.lower() for c in df.columns]
        df = df[['open', 'high', 'low', 'close', 'volume']]
        df['ticker'] = ticker
        
        rate_limiter.success()
        return df
        
    except Exception as e:
        rate_limiter.error()
        raise e


def validate_data(df, ticker):
    """Validate data quality and return issues."""
    issues = []
    
    # Check for missing values
    missing = df.isnull().sum().sum()
    if missing > 0:
        issues.append(f"⚠️ {missing} missing values")
    
    # Check date range
    days = len(df)
    if days < 100:
        issues.append(f"⚠️ Only {days} days of data (recommend 100+)")
    
    # Check for duplicates
    dupes = df.index.duplicated().sum()
    if dupes > 0:
        issues.append(f"⚠️ {dupes} duplicate dates")
    
    return issues


# Initialize
rate_limiter = RateLimiter(calls_per_minute=5)
all_data = {}

print("=" * 60)
print("📡 FETCHING MARKET DATA")
print("=" * 60)
print(f"\nSource: {CONFIG['data_source']}")
print(f"Tickers: {', '.join(CONFIG['tickers'])}")
print(f"Period: {CONFIG['start_date']} to {CONFIG['end_date']}")
print()

# Fetch data for each ticker
for ticker in tqdm(CONFIG['tickers'], desc="Downloading"):
    try:
        if CONFIG['verbose']:
            print(f"\n📥 Fetching {ticker}...")
        
        df = fetch_yahoo_data(
            ticker,
            CONFIG['start_date'],
            CONFIG['end_date'],
            rate_limiter
        )
        
        # Validate
        issues = validate_data(df, ticker)
        
        if CONFIG['verbose']:
            print(f"   ✅ {len(df)} rows downloaded")
            for issue in issues:
                print(f"   {issue}")
        
        all_data[ticker] = df
        
    except Exception as e:
        print(f"   ❌ Error fetching {ticker}: {e}")
        print(f"   💡 Check if ticker symbol is correct")

# Combine data
if all_data:
    DATA_RAW = pd.concat(all_data.values(), axis=0)
    
    print("\n" + "=" * 60)
    print("📊 DATA SUMMARY")
    print("=" * 60)
    print(f"\nTotal rows: {len(DATA_RAW):,}")
    print(f"Date range: {DATA_RAW.index.min().date()} to {DATA_RAW.index.max().date()}")
    print(f"Tickers: {DATA_RAW['ticker'].nunique()}")
    print(f"\nColumns: {list(DATA_RAW.columns)}")
    
    # Show preview
    print("\n📋 Preview (first 5 rows):")
    display(DATA_RAW.head())
    
    # Save checkpoint
    session.data = {'raw': DATA_RAW}
    session.state['data_loaded'] = True
    session.save_checkpoint('data_fetched', data_loaded=True)
    
    print("\n✅ Data fetched and saved!")
    print("   Proceed to Section 5 for preprocessing.")
else:
    print("\n❌ No data fetched. Please check your ticker symbols and try again.")

## 🔧 Section 5: Data Preprocessing

Transform raw price data into features suitable for neural network training.

### Preprocessing Steps:
1. **Technical Indicators**: RSI, MACD, Moving Averages, Bollinger Bands
2. **Normalization**: Scale features to 0-1 range for neural networks
3. **Sequence Creation**: Create lookback windows for time series
4. **Train/Test Split**: Separate data for training and evaluation

### Why This Matters:
- Neural networks work best with normalized data
- Technical indicators capture market patterns
- Proper splits prevent data leakage

In [ ]:
#@title 🔧 Preprocess Data { display-mode: "form" }
#@markdown Adds technical indicators, normalizes data, and creates train/test splits.

import numpy as np
from sklearn.preprocessing import MinMaxScaler

class DataPreprocessor:
    """Preprocess market data for neural network training."""
    
    def __init__(self, lookback=60):
        self.lookback = lookback
        self.scaler = MinMaxScaler()
        self.feature_columns = []
        
    def add_technical_indicators(self, df):
        """Add technical analysis indicators."""
        df = df.copy()
        
        # Moving Averages
        df['sma_10'] = df['close'].rolling(window=10).mean()
        df['sma_20'] = df['close'].rolling(window=20).mean()
        df['sma_50'] = df['close'].rolling(window=50).mean()
        df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()
        df['ema_26'] = df['close'].ewm(span=26, adjust=False).mean()
        
        # MACD
        df['macd'] = df['ema_12'] - df['ema_26']
        df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
        df['macd_hist'] = df['macd'] - df['macd_signal']
        
        # RSI (14-period)
        delta = df['close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        df['rsi'] = 100 - (100 / (1 + rs))
        
        # Bollinger Bands
        df['bb_middle'] = df['close'].rolling(window=20).mean()
        bb_std = df['close'].rolling(window=20).std()
        df['bb_upper'] = df['bb_middle'] + (bb_std * 2)
        df['bb_lower'] = df['bb_middle'] - (bb_std * 2)
        df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / df['bb_middle']
        
        # ATR (Average True Range)
        high_low = df['high'] - df['low']
        high_close = np.abs(df['high'] - df['close'].shift())
        low_close = np.abs(df['low'] - df['close'].shift())
        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        df['atr'] = tr.rolling(window=14).mean()
        
        # Volume indicators
        df['volume_sma'] = df['volume'].rolling(window=20).mean()
        df['volume_ratio'] = df['volume'] / df['volume_sma']
        
        # Price changes
        df['returns'] = df['close'].pct_change()
        df['log_returns'] = np.log(df['close'] / df['close'].shift(1))
        
        return df
    
    def normalize(self, df, fit=True):
        """Normalize numerical columns to 0-1 range."""
        self.feature_columns = [c for c in df.columns 
                               if c not in ['ticker'] and df[c].dtype in ['float64', 'int64']]
        
        if fit:
            df[self.feature_columns] = self.scaler.fit_transform(df[self.feature_columns])
        else:
            df[self.feature_columns] = self.scaler.transform(df[self.feature_columns])
        
        return df
    
    def create_sequences(self, df, target_col='close'):
        """Create sequences for time series prediction."""
        features = df[self.feature_columns].values
        target = df[target_col].values
        
        X, y = [], []
        for i in range(self.lookback, len(features)):
            X.append(features[i-self.lookback:i])
            y.append(target[i])
        
        return np.array(X), np.array(y)
    
    def process(self, df, verbose=True):
        """Full preprocessing pipeline."""
        if verbose:
            print("🔧 Preprocessing data...\n")
        
        # Step 1: Add indicators
        if verbose:
            print("   1️⃣ Adding technical indicators...")
        df = self.add_technical_indicators(df)
        
        # Step 2: Handle missing values
        if verbose:
            print("   2️⃣ Handling missing values...")
        initial_rows = len(df)
        df = df.dropna()
        dropped = initial_rows - len(df)
        if verbose and dropped > 0:
            print(f"      Dropped {dropped} rows with NaN values")
        
        # Step 3: Normalize
        if verbose:
            print("   3️⃣ Normalizing features...")
        df = self.normalize(df)
        
        # Step 4: Create sequences
        if verbose:
            print(f"   4️⃣ Creating sequences (lookback={self.lookback})...")
        X, y = self.create_sequences(df)
        
        if verbose:
            print(f"\n✅ Preprocessing complete!")
            print(f"   Features: {len(self.feature_columns)}")
            print(f"   Samples: {len(X)}")
            print(f"   Sequence shape: {X.shape}")
        
        return X, y, df


# Run preprocessing
preprocessor = DataPreprocessor(lookback=CONFIG['lookback'])

print("=" * 60)
print("🔧 DATA PREPROCESSING")
print("=" * 60)

X, y, df_processed = preprocessor.process(DATA_RAW, verbose=CONFIG['verbose'])

# Train/test split
split_idx = int(len(X) * CONFIG['train_split'])
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"\n📊 Train/Test Split ({CONFIG['train_split']:.0%}/{1-CONFIG['train_split']:.0%}):")
print(f"   Training: {len(X_train):,} samples")
print(f"   Testing:  {len(X_test):,} samples")

# Store for later use
session.data = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'feature_columns': preprocessor.feature_columns,
    'scaler': preprocessor.scaler
}
session.state['preprocessed'] = True

# Save checkpoint
session.save_checkpoint('preprocessed', preprocessed=True)

print("\n✅ Data preprocessed and saved!")
print("   Proceed to Section 6 for training.")

In [ ]:
#@title 📈 Visualize Preprocessed Data { display-mode: "form" }
#@markdown Optional: View data distributions and correlations.

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Feature distributions
ax1 = axes[0, 0]
sample_features = ['close', 'rsi', 'macd', 'volume_ratio']
available_features = [f for f in sample_features if f in df_processed.columns]
if available_features:
    df_processed[available_features[:4]].hist(ax=ax1, bins=50, alpha=0.7)
ax1.set_title('Feature Distributions (Normalized)')

# 2. Price with indicators
ax2 = axes[0, 1]
df_plot = df_processed.tail(200)
ax2.plot(df_plot.index, df_plot['close'], label='Close', alpha=0.8)
if 'sma_20' in df_plot.columns:
    ax2.plot(df_plot.index, df_plot['sma_20'], label='SMA 20', alpha=0.6)
if 'sma_50' in df_plot.columns:
    ax2.plot(df_plot.index, df_plot['sma_50'], label='SMA 50', alpha=0.6)
ax2.set_title('Price with Moving Averages (Last 200 days)')
ax2.legend()
ax2.tick_params(axis='x', rotation=45)

# 3. RSI
ax3 = axes[1, 0]
if 'rsi' in df_processed.columns:
    ax3.plot(df_plot.index, df_plot['rsi'], color='purple', alpha=0.8)
    ax3.axhline(y=0.7, color='r', linestyle='--', alpha=0.5, label='Overbought')
    ax3.axhline(y=0.3, color='g', linestyle='--', alpha=0.5, label='Oversold')
    ax3.set_title('RSI Indicator')
    ax3.legend()
    ax3.tick_params(axis='x', rotation=45)

# 4. Correlation heatmap
ax4 = axes[1, 1]
corr_cols = ['close', 'volume', 'rsi', 'macd', 'atr', 'bb_width']
corr_cols = [c for c in corr_cols if c in df_processed.columns]
if len(corr_cols) > 2:
    corr_matrix = df_processed[corr_cols].corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', 
                center=0, ax=ax4, cbar_kws={'shrink': 0.8})
    ax4.set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.show()

print("\n📊 Data visualization complete!")
print("   • Top-left: Feature distributions after normalization")
print("   • Top-right: Price trends with moving averages")
print("   • Bottom-left: RSI momentum indicator")
print("   • Bottom-right: Feature correlations")

## 🏋️ Section 6: Model Training

Now for the main event - training your neural network models!

### Available Models:
| Model | Description | Best For |
|-------|-------------|----------|
| **LSTM** | Long Short-Term Memory | Time series with long-term patterns |
| **DNN** | Deep Neural Network | Fast baseline, general patterns |
| **Transformer** | Attention-based | Complex pattern recognition |

### Training Process:
1. Model architecture is created based on your configuration
2. Training runs for specified epochs with progress display
3. Early stopping prevents overfitting
4. Checkpoints saved after training completes

In [ ]:
#@title 🧠 Define Neural Network Models { display-mode: "form" }
#@markdown PyTorch model architectures for LSTM, DNN, and Transformer.

import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    """LSTM model for time series prediction."""
    
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :])
        return out.squeeze()


class DNNModel(nn.Module):
    """Deep Neural Network for regression."""
    
    def __init__(self, input_dim, seq_len, hidden_dim=64, num_layers=3, dropout=0.2):
        super(DNNModel, self).__init__()
        
        flat_dim = input_dim * seq_len
        layers = []
        
        # Input layer
        layers.append(nn.Flatten())
        layers.append(nn.Linear(flat_dim, hidden_dim * 2))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        
        # Hidden layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim * 2, hidden_dim * 2))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
        
        # Output layer
        layers.append(nn.Linear(hidden_dim * 2, 1))
        
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x).squeeze()


class TransformerModel(nn.Module):
    """Transformer model for time series prediction."""
    
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, nhead=4, dropout=0.2):
        super(TransformerModel, self).__init__()
        
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=nhead,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        
    def forward(self, x):
        x = self.input_projection(x)
        x = self.transformer(x)
        out = self.fc(x[:, -1, :])
        return out.squeeze()


def create_model(model_type, input_dim, seq_len, config):
    """Factory function to create model by type."""
    if model_type == 'LSTM':
        return LSTMModel(
            input_dim=input_dim,
            hidden_dim=config['hidden_dim'],
            num_layers=config['num_layers'],
            dropout=config['dropout']
        )
    elif model_type == 'DNN':
        return DNNModel(
            input_dim=input_dim,
            seq_len=seq_len,
            hidden_dim=config['hidden_dim'],
            num_layers=config['num_layers'],
            dropout=config['dropout']
        )
    elif model_type == 'Transformer':
        return TransformerModel(
            input_dim=input_dim,
            hidden_dim=config['hidden_dim'],
            num_layers=config['num_layers'],
            dropout=config['dropout']
        )
    else:
        raise ValueError(f"Unknown model type: {model_type}")

print("✅ Model architectures defined!")
print("\n📋 Available models:")
print("   • LSTM - Long Short-Term Memory network")
print("   • DNN - Deep Neural Network")
print("   • Transformer - Attention-based model")

In [ ]:
#@title 🏋️ Train Models { display-mode: "form" }
#@markdown Trains selected models with progress tracking and early stopping.

from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm
import copy

class Trainer:
    """Training manager with progress tracking and early stopping."""
    
    def __init__(self, model, device, config, verbose=True):
        self.model = model.to(device)
        self.device = device
        self.config = config
        self.verbose = verbose
        self.history = {'train_loss': [], 'val_loss': []}
        self.best_model = None
        self.best_loss = float('inf')
        
    def train(self, train_loader, val_loader, epochs, patience=10):
        """Train the model with early stopping."""
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.config['learning_rate'])
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
        
        patience_counter = 0
        
        # Progress bar
        epoch_iter = tqdm(range(epochs), desc="Training", disable=not self.verbose)
        
        for epoch in epoch_iter:
            # Training phase
            self.model.train()
            train_losses = []
            
            for X_batch, y_batch in train_loader:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)
                
                optimizer.zero_grad()
                predictions = self.model(X_batch)
                loss = criterion(predictions, y_batch)
                loss.backward()
                optimizer.step()
                
                train_losses.append(loss.item())
            
            # Validation phase
            self.model.eval()
            val_losses = []
            
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch = X_batch.to(self.device)
                    y_batch = y_batch.to(self.device)
                    
                    predictions = self.model(X_batch)
                    loss = criterion(predictions, y_batch)
                    val_losses.append(loss.item())
            
            # Calculate epoch metrics
            train_loss = np.mean(train_losses)
            val_loss = np.mean(val_losses)
            
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            
            # Update scheduler
            scheduler.step(val_loss)
            
            # Update progress bar
            epoch_iter.set_postfix({
                'train_loss': f'{train_loss:.4f}',
                'val_loss': f'{val_loss:.4f}'
            })
            
            # Early stopping check
            if val_loss < self.best_loss:
                self.best_loss = val_loss
                self.best_model = copy.deepcopy(self.model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                
            if patience_counter >= patience:
                if self.verbose:
                    print(f"\n⏹️ Early stopping at epoch {epoch+1}")
                break
        
        # Restore best model
        if self.best_model:
            self.model.load_state_dict(self.best_model)
        
        return self.history


# Prepare data loaders
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

# Store trained models
TRAINED_MODELS = {}
TRAINING_HISTORIES = {}

print("=" * 60)
print("🏋️ MODEL TRAINING")
print("=" * 60)

for model_name in CONFIG['models']:
    print(f"\n{'='*60}")
    print(f"🤖 Training {model_name}")
    print(f"{'='*60}")
    
    # Create model
    input_dim = X_train.shape[2]
    seq_len = X_train.shape[1]
    
    model = create_model(model_name, input_dim, seq_len, CONFIG)
    
    # Count parameters
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   Parameters: {params:,}")
    print(f"   Device: {DEVICE}")
    print()
    
    # Train
    trainer = Trainer(model, DEVICE, CONFIG, verbose=CONFIG['verbose'])
    history = trainer.train(train_loader, val_loader, CONFIG['epochs'], patience=15)
    
    # Store results
    TRAINED_MODELS[model_name] = trainer.model
    TRAINING_HISTORIES[model_name] = history
    
    print(f"\n✅ {model_name} training complete!")
    print(f"   Best validation loss: {trainer.best_loss:.4f}")

# Save checkpoint
session.state['models_trained'] = list(TRAINED_MODELS.keys())
session.save_checkpoint('trained', models_trained=list(TRAINED_MODELS.keys()))

print("\n" + "=" * 60)
print(f"🎉 Training complete! {len(TRAINED_MODELS)} model(s) trained.")
print("   Proceed to Section 7 for evaluation.")

In [ ]:
#@title 📉 Visualize Training History { display-mode: "form" }
#@markdown Plot training and validation loss curves for all models.

fig, axes = plt.subplots(1, len(TRAINING_HISTORIES), figsize=(6*len(TRAINING_HISTORIES), 4))

if len(TRAINING_HISTORIES) == 1:
    axes = [axes]

for ax, (model_name, history) in zip(axes, TRAINING_HISTORIES.items()):
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    ax.plot(epochs_range, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
    ax.plot(epochs_range, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    
    # Mark best epoch
    best_epoch = np.argmin(history['val_loss']) + 1
    best_val = min(history['val_loss'])
    ax.axvline(x=best_epoch, color='g', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.set_title(f'{model_name} Training History')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Training Analysis:")
for model_name, history in TRAINING_HISTORIES.items():
    final_train = history['train_loss'][-1]
    final_val = history['val_loss'][-1]
    best_val = min(history['val_loss'])
    
    print(f"\n{model_name}:")
    print(f"   Final Train Loss: {final_train:.4f}")
    print(f"   Final Val Loss:   {final_val:.4f}")
    print(f"   Best Val Loss:    {best_val:.4f}")
    
    # Check for overfitting
    if final_train < final_val * 0.5:
        print(f"   ⚠️ Possible overfitting detected")

## 🔍 Section 7: Hyperparameter Tuning (Optional)

Use automated hyperparameter optimization to find the best model configuration.

### Available Methods:
- **Grid Search**: Try all combinations (thorough but slow)
- **Random Search**: Sample random combinations (faster)
- **Optuna**: Intelligent search with pruning (recommended)

*Skip this section if you're happy with current results.*

In [ ]:
#@title 🔍 Hyperparameter Tuning with Optuna { display-mode: "form" }
#@markdown Automatically find optimal hyperparameters using Bayesian optimization.

import optuna
from optuna.trial import TrialState

# Configuration
TUNE_MODEL = 'LSTM'  #@param ["LSTM", "DNN", "Transformer"]
N_TRIALS = 20  #@param {type:"integer"}
TUNE_EPOCHS = 30  # Fewer epochs for faster tuning

def objective(trial):
    """Optuna objective function for hyperparameter optimization."""
    
    # Suggest hyperparameters
    hidden_dim = trial.suggest_categorical('hidden_dim', [32, 64, 128, 256])
    num_layers = trial.suggest_int('num_layers', 1, 4)
    dropout = trial.suggest_float('dropout', 0.0, 0.5, step=0.1)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    
    # Create model with suggested params
    config_trial = {
        'hidden_dim': hidden_dim,
        'num_layers': num_layers,
        'dropout': dropout,
        'learning_rate': learning_rate,
        'batch_size': batch_size
    }
    
    input_dim = X_train.shape[2]
    seq_len = X_train.shape[1]
    
    model = create_model(TUNE_MODEL, input_dim, seq_len, config_trial)
    model = model.to(DEVICE)
    
    # Create data loaders with trial batch size
    train_loader_trial = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader_trial = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Training
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    best_val_loss = float('inf')
    
    for epoch in range(TUNE_EPOCHS):
        # Train
        model.train()
        for X_batch, y_batch in train_loader_trial:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        
        # Validate
        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader_trial:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                val_losses.append(criterion(model(X_batch), y_batch).item())
        
        val_loss = np.mean(val_losses)
        best_val_loss = min(best_val_loss, val_loss)
        
        # Report for pruning
        trial.report(val_loss, epoch)
        
        if trial.should_prune():
            raise optuna.TrialPruned()
    
    return best_val_loss


print("=" * 60)
print(f"🔍 HYPERPARAMETER TUNING - {TUNE_MODEL}")
print("=" * 60)
print(f"\nTrials: {N_TRIALS}")
print(f"Epochs per trial: {TUNE_EPOCHS}")
print("\nSearching for optimal configuration...\n")

# Create and run study
study = optuna.create_study(
    direction='minimize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# Results
print("\n" + "=" * 60)
print("📊 TUNING RESULTS")
print("=" * 60)

print(f"\n🏆 Best Trial:")
print(f"   Value (Val Loss): {study.best_trial.value:.4f}")
print(f"\n⚙️ Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"   {key}: {value}")

# Store best params
BEST_PARAMS = study.best_params.copy()

print("\n💡 To use these parameters, update CONFIG and retrain:")
print("   CONFIG['hidden_dim'] =", BEST_PARAMS.get('hidden_dim'))
print("   CONFIG['num_layers'] =", BEST_PARAMS.get('num_layers'))
print("   CONFIG['dropout'] =", BEST_PARAMS.get('dropout'))
print("   CONFIG['learning_rate'] =", BEST_PARAMS.get('learning_rate'))

## 📊 Section 8: Model Evaluation

Evaluate trained models with comprehensive metrics and backtesting.

### Evaluation Includes:
- **Regression Metrics**: MAE, RMSE, R², MAPE
- **Trading Metrics**: Sharpe Ratio, Max Drawdown, Win Rate
- **Backtesting**: Simulated trading performance
- **Visualizations**: Predictions vs actual, equity curves

In [ ]:
#@title 📊 Evaluate Models { display-mode: "form" }
#@markdown Calculate metrics and run backtesting for all trained models.

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

class ModelEvaluator:
    """Comprehensive model evaluation with trading metrics."""
    
    def __init__(self, model, device):
        self.model = model
        self.device = device
        
    def predict(self, X):
        """Generate predictions."""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.model(X_tensor).cpu().numpy()
        return predictions
    
    def calculate_metrics(self, y_true, y_pred):
        """Calculate regression metrics."""
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        r2 = r2_score(y_true, y_pred)
        mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
        
        return {
            'MAE': mae,
            'RMSE': rmse,
            'R²': r2,
            'MAPE (%)': mape
        }
    
    def calculate_trading_metrics(self, y_true, y_pred):
        """Calculate trading-specific metrics."""
        # Calculate returns based on predictions
        pred_direction = np.sign(np.diff(y_pred, prepend=y_pred[0]))
        actual_returns = np.diff(y_true, prepend=y_true[0]) / (y_true + 1e-8)
        
        # Strategy returns (long when predicting up)
        strategy_returns = pred_direction[1:] * actual_returns[1:]
        
        # Sharpe Ratio (annualized)
        if np.std(strategy_returns) > 0:
            sharpe = np.mean(strategy_returns) / np.std(strategy_returns) * np.sqrt(252)
        else:
            sharpe = 0
        
        # Max Drawdown
        cumulative = np.cumprod(1 + strategy_returns)
        running_max = np.maximum.accumulate(cumulative)
        drawdown = (cumulative - running_max) / running_max
        max_drawdown = np.min(drawdown) * 100
        
        # Win Rate
        win_rate = np.mean(strategy_returns > 0) * 100
        
        # Total Return
        total_return = (np.prod(1 + strategy_returns) - 1) * 100
        
        return {
            'Sharpe Ratio': sharpe,
            'Max Drawdown (%)': max_drawdown,
            'Win Rate (%)': win_rate,
            'Total Return (%)': total_return
        }
    
    def backtest(self, y_true, y_pred):
        """Run simple backtest."""
        pred_direction = np.sign(np.diff(y_pred, prepend=y_pred[0]))
        actual_returns = np.diff(y_true, prepend=y_true[0]) / (y_true + 1e-8)
        strategy_returns = pred_direction[1:] * actual_returns[1:]
        
        equity_curve = np.cumprod(1 + strategy_returns)
        buy_hold = np.cumprod(1 + actual_returns[1:])
        
        return {
            'strategy': equity_curve,
            'buy_hold': buy_hold,
            'returns': strategy_returns
        }


# Evaluate all models
EVALUATION_RESULTS = {}

print("=" * 60)
print("📊 MODEL EVALUATION")
print("=" * 60)

for model_name, model in TRAINED_MODELS.items():
    print(f"\n{'='*60}")
    print(f"🔍 Evaluating {model_name}")
    print(f"{'='*60}")
    
    evaluator = ModelEvaluator(model, DEVICE)
    
    # Generate predictions
    y_pred = evaluator.predict(X_test)
    
    # Calculate metrics
    regression_metrics = evaluator.calculate_metrics(y_test, y_pred)
    trading_metrics = evaluator.calculate_trading_metrics(y_test, y_pred)
    backtest_results = evaluator.backtest(y_test, y_pred)
    
    # Store results
    EVALUATION_RESULTS[model_name] = {
        'predictions': y_pred,
        'regression': regression_metrics,
        'trading': trading_metrics,
        'backtest': backtest_results
    }
    
    # Display metrics
    print("\n📈 Regression Metrics:")
    for metric, value in regression_metrics.items():
        print(f"   {metric}: {value:.4f}")
    
    print("\n💰 Trading Metrics:")
    for metric, value in trading_metrics.items():
        print(f"   {metric}: {value:.2f}")

# Summary comparison
print("\n" + "=" * 60)
print("📊 MODEL COMPARISON")
print("=" * 60)

comparison_data = []
for model_name, results in EVALUATION_RESULTS.items():
    row = {'Model': model_name}
    row.update(results['regression'])
    row.update(results['trading'])
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# Find best model
best_sharpe_model = max(EVALUATION_RESULTS.keys(), 
                        key=lambda x: EVALUATION_RESULTS[x]['trading']['Sharpe Ratio'])
print(f"\n🏆 Best model by Sharpe Ratio: {best_sharpe_model}")

In [ ]:
#@title 📈 Visualize Evaluation Results { display-mode: "form" }
#@markdown Plot predictions, equity curves, and model comparisons.

n_models = len(EVALUATION_RESULTS)
fig, axes = plt.subplots(n_models, 3, figsize=(16, 5*n_models))

if n_models == 1:
    axes = axes.reshape(1, -1)

for idx, (model_name, results) in enumerate(EVALUATION_RESULTS.items()):
    y_pred = results['predictions']
    backtest = results['backtest']
    
    # 1. Predictions vs Actual
    ax1 = axes[idx, 0]
    ax1.plot(y_test[-200:], 'b-', label='Actual', alpha=0.7)
    ax1.plot(y_pred[-200:], 'r-', label='Predicted', alpha=0.7)
    ax1.set_title(f'{model_name}: Predictions vs Actual (Last 200)')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Normalized Price')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Scatter plot
    ax2 = axes[idx, 1]
    ax2.scatter(y_test, y_pred, alpha=0.3, s=10)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax2.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect')
    ax2.set_title(f'{model_name}: Prediction Scatter')
    ax2.set_xlabel('Actual')
    ax2.set_ylabel('Predicted')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Equity curve
    ax3 = axes[idx, 2]
    ax3.plot(backtest['strategy'], 'g-', label='Strategy', linewidth=2)
    ax3.plot(backtest['buy_hold'], 'b--', label='Buy & Hold', linewidth=2, alpha=0.7)
    ax3.set_title(f'{model_name}: Backtest Equity Curve')
    ax3.set_xlabel('Time')
    ax3.set_ylabel('Portfolio Value')
    ax3.legend()
    ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Returns distribution
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (model_name, results) in zip(axes, EVALUATION_RESULTS.items()):
    returns = results['backtest']['returns']
    ax.hist(returns, bins=50, alpha=0.7, color='steelblue', edgecolor='white')
    ax.axvline(0, color='red', linestyle='--')
    ax.axvline(np.mean(returns), color='green', linestyle='--', label=f'Mean: {np.mean(returns):.4f}')
    ax.set_title(f'{model_name}: Returns Distribution')
    ax.set_xlabel('Return')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.show()

print("✅ Evaluation visualizations complete!")

## 🚀 Section 9: Export to ONNX

Export trained models to ONNX format for deployment to the dashboard.

### Why ONNX?
- **Universal format**: Works across different frameworks
- **Optimized inference**: Faster prediction in production
- **Dashboard compatible**: Ready for upload to your Next.js dashboard

### Export Process:
1. Convert PyTorch model to ONNX
2. Validate exported model
3. Save to Google Drive
4. Download for dashboard upload

In [ ]:
#@title 📦 Export Models to ONNX { display-mode: "form" }
#@markdown Export trained models for dashboard deployment.

import onnx
import onnxruntime as ort

class ONNXExporter:
    """Export PyTorch models to ONNX format."""
    
    def __init__(self, export_dir):
        self.export_dir = export_dir
        os.makedirs(export_dir, exist_ok=True)
        
    def export(self, model, model_name, input_shape, config):
        """Export model to ONNX format."""
        model.eval()
        
        # Create dummy input
        dummy_input = torch.randn(1, *input_shape)
        
        # Generate filename with metadata
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'{model_name}_{timestamp}.onnx'
        filepath = os.path.join(self.export_dir, filename)
        
        # Export to ONNX
        torch.onnx.export(
            model.cpu(),
            dummy_input,
            filepath,
            export_params=True,
            opset_version=14,
            do_constant_folding=True,
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={
                'input': {0: 'batch_size'},
                'output': {0: 'batch_size'}
            }
        )
        
        # Validate
        onnx_model = onnx.load(filepath)
        onnx.checker.check_model(onnx_model)
        
        # Add metadata
        meta = onnx_model.metadata_props.add()
        meta.key = 'model_type'
        meta.value = model_name
        
        meta = onnx_model.metadata_props.add()
        meta.key = 'config'
        meta.value = json.dumps(config)
        
        meta = onnx_model.metadata_props.add()
        meta.key = 'export_timestamp'
        meta.value = timestamp
        
        onnx.save(onnx_model, filepath)
        
        # Get file size
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        
        return {
            'filepath': filepath,
            'filename': filename,
            'size_mb': size_mb
        }
    
    def validate(self, filepath, sample_input):
        """Validate ONNX model with sample inference."""
        session = ort.InferenceSession(filepath)
        
        input_name = session.get_inputs()[0].name
        output = session.run(None, {input_name: sample_input.astype(np.float32)})
        
        return output[0]


# Export all trained models
exporter = ONNXExporter(DIRS['exports'])
EXPORTED_MODELS = {}

print("=" * 60)
print("📦 EXPORTING MODELS TO ONNX")
print("=" * 60)

input_shape = (CONFIG['lookback'], X_train.shape[2])
sample_input = X_test[:1]

for model_name, model in TRAINED_MODELS.items():
    print(f"\n🔄 Exporting {model_name}...")
    
    try:
        # Export
        result = exporter.export(model, model_name, input_shape, CONFIG)
        
        print(f"   ✅ Exported: {result['filename']}")
        print(f"   📁 Path: {result['filepath']}")
        print(f"   💾 Size: {result['size_mb']:.2f} MB")
        
        # Validate
        print("   🔍 Validating...")
        onnx_pred = exporter.validate(result['filepath'], sample_input)
        
        # Compare with PyTorch prediction
        model.eval()
        with torch.no_grad():
            torch_pred = model.cpu()(torch.FloatTensor(sample_input)).numpy()
        
        diff = np.abs(onnx_pred.flatten() - torch_pred.flatten()).max()
        print(f"   ✅ Validation passed (max diff: {diff:.6f})")
        
        EXPORTED_MODELS[model_name] = result
        
    except Exception as e:
        print(f"   ❌ Export failed: {e}")

# Summary
print("\n" + "=" * 60)
print("📋 EXPORT SUMMARY")
print("=" * 60)

for model_name, result in EXPORTED_MODELS.items():
    print(f"\n{model_name}:")
    print(f"   File: {result['filename']}")
    print(f"   Size: {result['size_mb']:.2f} MB")
    print(f"   Path: {result['filepath']}")

print(f"\n✅ {len(EXPORTED_MODELS)} model(s) exported successfully!")
print(f"📁 Export folder: {DIRS['exports']}")

In [ ]:
#@title ⬇️ Download ONNX Files { display-mode: "form" }
#@markdown Download exported models to your local machine.

if IN_COLAB:
    from google.colab import files
    
    print("=" * 60)
    print("⬇️ DOWNLOAD ONNX FILES")
    print("=" * 60)
    
    for model_name, result in EXPORTED_MODELS.items():
        print(f"\n📥 Downloading {model_name}...")
        try:
            files.download(result['filepath'])
            print(f"   ✅ Download started: {result['filename']}")
        except Exception as e:
            print(f"   ⚠️ Download failed: {e}")
            print(f"   💡 You can manually download from: {result['filepath']}")
    
    print("\n✅ Downloads initiated!")
    print("   Check your browser's download folder.")
else:
    print("📁 Running locally - files are already saved to:")
    for model_name, result in EXPORTED_MODELS.items():
        print(f"   {result['filepath']}")

## 📡 Section 10: Experiment Tracking

Log experiments to W&B (Weights & Biases) and/or your dashboard API for tracking and comparison.

### What Gets Logged:
- Experiment configuration
- Training metrics (loss per epoch)
- Evaluation results
- Model metadata

*This section is optional if you haven't configured API keys.*

In [ ]:
#@title 📡 Log to Experiment Trackers { display-mode: "form" }
#@markdown Log experiments to W&B and/or dashboard API.

import requests

class ExperimentLogger:
    """Log experiments to multiple tracking services."""
    
    def __init__(self, config, api_keys):
        self.config = config
        self.api_keys = api_keys
        self.experiment_id = f"{config['experiment_name']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
    def log_to_wandb(self, training_histories, evaluation_results):
        """Log to Weights & Biases."""
        if not self.api_keys.get('wandb'):
            print("⚪ W&B not configured - skipping")
            return False
        
        try:
            import wandb
            
            wandb.login(key=self.api_keys['wandb'])
            
            for model_name in training_histories.keys():
                run = wandb.init(
                    project='trading-ml',
                    name=f"{self.experiment_id}_{model_name}",
                    config=self.config,
                    reinit=True
                )
                
                # Log training history
                history = training_histories[model_name]
                for epoch in range(len(history['train_loss'])):
                    wandb.log({
                        'train_loss': history['train_loss'][epoch],
                        'val_loss': history['val_loss'][epoch],
                        'epoch': epoch + 1
                    })
                
                # Log evaluation metrics
                if model_name in evaluation_results:
                    wandb.log(evaluation_results[model_name]['regression'])
                    wandb.log(evaluation_results[model_name]['trading'])
                
                run.finish()
            
            print("✅ Logged to W&B")
            return True
            
        except Exception as e:
            print(f"⚠️ W&B logging failed: {e}")
            return False
    
    def log_to_dashboard(self, training_histories, evaluation_results, exported_models):
        """Log to dashboard API."""
        if not self.api_keys.get('dashboard'):
            print("⚪ Dashboard API not configured - skipping")
            return False
        
        # Dashboard API URL (update with your actual endpoint)
        api_url = os.environ.get('DASHBOARD_API_URL', 'http://localhost:3000/api')
        
        try:
            for model_name in training_histories.keys():
                payload = {
                    'experiment_id': self.experiment_id,
                    'model_type': model_name,
                    'config': self.config,
                    'training_history': training_histories[model_name],
                    'evaluation': evaluation_results.get(model_name, {}),
                    'export_info': exported_models.get(model_name, {}),
                    'timestamp': datetime.now().isoformat()
                }
                
                response = requests.post(
                    f"{api_url}/experiments",
                    json=payload,
                    headers={
                        'Authorization': f"Bearer {self.api_keys['dashboard']}",
                        'Content-Type': 'application/json'
                    },
                    timeout=30
                )
                
                if response.status_code in [200, 201]:
                    print(f"✅ {model_name} logged to dashboard")
                else:
                    print(f"⚠️ Dashboard API returned {response.status_code}")
            
            return True
            
        except requests.exceptions.ConnectionError:
            print("⚠️ Could not connect to dashboard API")
            print("   💡 Make sure your dashboard is running")
            return False
        except Exception as e:
            print(f"⚠️ Dashboard logging failed: {e}")
            return False


# Initialize logger
logger = ExperimentLogger(CONFIG, API_KEYS)

print("=" * 60)
print("📡 EXPERIMENT TRACKING")
print("=" * 60)

print(f"\n🏷️ Experiment ID: {logger.experiment_id}")

# Log to W&B
print("\n1️⃣ Weights & Biases:")
logger.log_to_wandb(TRAINING_HISTORIES, EVALUATION_RESULTS)

# Log to Dashboard
print("\n2️⃣ Dashboard API:")
logger.log_to_dashboard(TRAINING_HISTORIES, EVALUATION_RESULTS, EXPORTED_MODELS)

# Save experiment summary locally
summary = {
    'experiment_id': logger.experiment_id,
    'config': CONFIG,
    'models': list(TRAINED_MODELS.keys()),
    'evaluation': {k: {**v['regression'], **v['trading']} for k, v in EVALUATION_RESULTS.items()},
    'exports': {k: v['filename'] for k, v in EXPORTED_MODELS.items()},
    'timestamp': datetime.now().isoformat()
}

summary_path = os.path.join(DIRS['logs'], f'{logger.experiment_id}_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\n💾 Experiment summary saved: {summary_path}")
print("\n✅ Experiment tracking complete!")

## 🎉 Section 11: Summary & Next Steps

Congratulations! You've completed the neural network training pipeline.

### What You've Accomplished:
- ✅ Set up your environment and API keys
- ✅ Configured experiment parameters
- ✅ Fetched and preprocessed market data
- ✅ Trained neural network model(s)
- ✅ Evaluated performance with backtesting
- ✅ Exported models to ONNX format
- ✅ Logged experiments for tracking

### Next Steps:
1. **Upload to Dashboard**: Go to your Next.js dashboard and upload the ONNX files
2. **Paper Trading**: Test models with simulated trading before going live
3. **Try Different Models**: Experiment with Tree-Based models notebook
4. **Improve Performance**: Tune hyperparameters or try different features

In [ ]:
#@title 📋 Final Summary { display-mode: "form" }
#@markdown Display complete summary of your experiment.

print("=" * 60)
print("🎉 EXPERIMENT COMPLETE!")
print("=" * 60)

print(f"\n🏷️ Experiment: {CONFIG['experiment_name']}")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

print("\n📊 DATA:")
print(f"   Tickers: {', '.join(CONFIG['tickers'])}")
print(f"   Period: {CONFIG['start_date']} to {CONFIG['end_date']}")
print(f"   Samples: {len(X_train) + len(X_test):,}")

print("\n🤖 MODELS TRAINED:")
for model_name in TRAINED_MODELS.keys():
    results = EVALUATION_RESULTS.get(model_name, {})
    trading = results.get('trading', {})
    print(f"\n   {model_name}:")
    print(f"      Sharpe Ratio: {trading.get('Sharpe Ratio', 'N/A'):.2f}")
    print(f"      Max Drawdown: {trading.get('Max Drawdown (%)', 'N/A'):.2f}%")
    print(f"      Win Rate: {trading.get('Win Rate (%)', 'N/A'):.2f}%")

print("\n📦 EXPORTED FILES:")
for model_name, export in EXPORTED_MODELS.items():
    print(f"   {export['filename']} ({export['size_mb']:.2f} MB)")

print("\n📁 FILES SAVED TO:")
print(f"   Checkpoints: {DIRS['checkpoints']}")
print(f"   Exports: {DIRS['exports']}")
print(f"   Logs: {DIRS['logs']}")

print("\n" + "=" * 60)
print("🚀 NEXT STEPS:")
print("=" * 60)
print("""
1. 📤 Upload ONNX files to your dashboard:
   - Go to your dashboard URL
   - Navigate to 'Model Management'
   - Drag and drop ONNX files

2. 📈 Start paper trading to validate performance

3. 🔄 Try the Tree-Based Models notebook for comparison

4. 📚 Review experiment logs in W&B or dashboard
""")

print("✨ Thank you for using the Neural Networks Training Pipeline!")